# Phase 5 & 6: Data Preparation (Track A & Track B Sync)
**Project:** AI Stock Movement Prediction & Backtesting Platform  
**Objectives:**
1. **Phase 5:** Create the Target Variable `target` which is `1` if Next-Day Forward Return > 0, else `0`. Ensure no look-ahead bias.
2. **Phase 6:** Implement a strict chronological Time-Series Split into Train (2018-2023), Validation (2024), and Test (2025-2026) sets.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
from src.data.preparation import create_target_variable, split_time_series

print("Loaded preparation modules.")

## 1. Load Cleaned Market Data

In [ ]:
df = pd.read_csv("../data/processed/combined_market_data.csv")
print(f"Initial shape: {df.shape}")

## 2. Phase 5: Target Variable Creation
Notice how `forward_return_1d` is computed strictly using `shift(-1)` per ticker. We will drop the last missing day for training (`is_live=False`).

In [ ]:
df_with_target = create_target_variable(df, is_live=False)

print(f"Shape after target creation (last row dropped): {df_with_target.shape}")
df_with_target[['date', 'ticker', 'adj_close', 'forward_return_1d', 'target']].tail()

## 3. Phase 6: Time-Series Split
We use strict date boundaries to completely avoid any overlap between training patterns and future validation/test data.

In [ ]:
train_df, val_df, test_df = split_time_series(df_with_target)

print("\n--- Time-Series Audit ---")
print(f"Train max date: {train_df['date'].max()} -> Val min date: {val_df['date'].min()}")
print(f"Val max date  : {val_df['date'].max()} -> Test min date: {test_df['date'].min()}")

## 4. Class Balance Check in Train Set
Evaluating if we need SMOTE or class weights. A 50-50 balance implies accuracy and ROC-AUC will naturally align well without heavy class resampling.

In [ ]:
print("Train Target Class Balance:")
print(train_df['target'].value_counts(normalize=True) * 100)